# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass
import os

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [2]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_URL = getpass("Elastic URL: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
    index_name="kayode_index_elas", #give it a meaningful name,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [3]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [4]:
pip install jq

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    None

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=300 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [6]:

documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="kayode_index_elas",
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [9]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


---- Answer ----

The NASA sales team is a part of the Americas region in the company's sales organization. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services in the North America and South America regions. They work closely with other departments, such as marketing, product development, and customer support, to ensure the company's success in these regions.


In [10]:


ans = chain.invoke("Tell me this document?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide me with this document?', '2. What information can you give me about this document?', '3. Is there any relevant document related to this topic?']


---- Answer ----

This document is a sales strategy for a tech company, outlining their objectives, focus areas, and action plans for the fiscal year 2024. It includes goals such as increasing revenue and market share, retaining customers, and launching new products. The strategy involves developing the sales team, marketing and promotion, building partnerships, and focusing on customer success. The company will also regularly monitor and evaluate their progress to ensure alignment with market trends.


In [11]:
ans = chain.invoke("Are you human?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is your nature?', '2. Can you pass for a human?', '3. Are you a machine or a person?']


---- Answer ----
No, I am an AI assistant designed to help with question-answering tasks. I am not a human, but I am programmed to provide helpful and informative responses. Is there something specific you would like to know?


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [ ]:
ans = chain.invoke("Can you talk abot everything?")


print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on a wide range of topics?', '2. Is there anything you cannot discuss?', '3. What subjects are you knowledgeable about?']
INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the technical specifications for Amazon products?', "2. Can you provide information on the tech specs of Amazon's offerings?", "3. How can you describe the technical details of Amazon's technology?"]


---- Answer ----

I am an AI assistant and do not have access to specific information about Amazon's tech specs. However, Amazon is a large and diverse company with a wide range of products and services, so it would be difficult to cover all of their tech specs in one answer. Is there a specific product or service you are interested in learning about? I may be able to provide more specific information if you have a specific question.


In [14]:
ans = chain.invoke("Can you talk abot everything Amazon tech specs?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the technical specifications for Amazon products?', "2. Can you provide information on the tech specs of Amazon's offerings?", "3. How can you describe the technical details of Amazon's technology?"]


---- Answer ----

I am an AI assistant and do not have access to specific information about Amazon's tech specs. However, Amazon is a large and diverse company with a wide range of products and services, so it would be difficult to cover all of their tech specs in one answer. Is there a specific product or service you are interested in learning about? I may be able to provide more specific information if you have a specific question.


In [15]:
ans = chain.invoke("List what you can talk about")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What topics can you discuss?', '2. Can you provide a list of subjects you are knowledgeable about?', '3. What are some of the things you are able to converse about?']


---- Answer ----

I can talk about our company culture and values, the key steps for onboarding, updating tax elections and documents, benefits enrollment, and getting settled in your workspace. I can also discuss our hybrid work policy, the career leveling matrix for Software Engineers, and how we support employee growth and development.


In [16]:
ans = chain.invoke("Which company?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the name of the company?', "2. Can you provide me with the company's name?", '3. Which corporation are you referring to?']


---- Answer ----
The company implementing a two-day in-office work requirement per week for all eligible employees starting May 2022.


In [17]:
ans = chain.invoke("You are confusing me , what comany re you refering to?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you clarify which company you are referring to? ', "2. I'm having trouble understanding, can you specify the company in question? ", '3. Could you provide more context and specify the company you are talking about?']


---- Answer ----
I apologize for any confusion. The context provided is referring to a hypothetical company that is implementing a new hybrid work policy. This policy requires eligible employees to work in the office for a certain number of days per week, while also allowing for remote work. The specific details and dates mentioned in the context are not related to any specific company, but rather serve as an example for how a company may communicate and implement a new work policy. 


In [18]:
ans = chain.invoke("Then why did you answer for the NASA sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What was the reason for your response to the NASA sales team?', '2. Can you explain why you answered on behalf of the NASA sales team?', '3. What motivated you to answer for the NASA sales team?']


---- Answer ----

I am an AI assistant designed to answer questions based on retrieved context. In this context, the NASA sales team was mentioned as part of the overall sales organization structure. As an AI, I do not have personal opinions or preferences, and I simply provide information based on the given context.


In [19]:
ans = chain.invoke("Can your desingers give you a name instead of just calling you an assistant? ")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can your designers provide you with a unique name instead of simply referring to you as an assistant?', '2. Is it possible for your designers to give you a personalized name rather than just using the generic term "assistant"?', '3. Can your designers assign you a specific name instead of just calling you an assistant?']


---- Answer ----

I am an AI assistant designed to assist with question-answering tasks. I do not have a physical form or a name, but I am always here to help with any questions you may have. Is there something specific you would like to know?


In [21]:
ans = chain.invoke("What infomation do you have , give me a bullet point summary")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide a bullet point summary of the information you have?', '2. What are the key points of the information you possess?', '3. Could you give me a summary of the information in bullet points?']


---- Answer ----
- The sales strategy document outlines the objectives, focus areas, and action plans for a tech company's sales operations in fiscal year 2024.
     - The primary goal is to increase revenue, expand market share, and strengthen customer relationships in target markets.
     - Objectives for fiscal year 2024 include increasing revenue by 20%, expanding market share by 15%, retaining 95% of existing customers, and launching at least two new products or services.
     - Focus areas include targeting high-growth industries, strengthening relationships with key accounts, and optimizing the product/service portfolio.
     - Action plans include expanding the sales team, developing targeted marketing campaigns, and implementing a proactive customer success program.
     - The strategy will be monitored and evaluated through key performance indicators and regular reviews.
